# GOAL
#### Can a simple bag-of-words + logistic regression model classify folktales by nation?

In [164]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from pathlib import Path

data_path = Path("../data/processed/folktales_model_ready.csv")

df = pd.read_csv(data_path)

df.head(3)

,source_url,nation,title,text,char_count,word_count,clean_text
0,https://fairytalez.com/sharing-joy-and-sorrow/,german,Sharing Joy and Sorrow,"There was once a tailor, who was a quarrelsome...",1967,377,there was once a tailor who was a quarrelsome ...
1,https://fairytalez.com/the-punishment-of-gangana/,french,The Punishment of Gangana,Once upon a time there lived a king and queen ...,23188,4306,once upon a time there lived a king and queen ...
2,https://fairytalez.com/the-peace-with-the-snakes/,north_american_native,The Peace with the Snakes,In those days there was a Piegan chief named O...,16050,3122,in those days there was a piegan chief named o...


### Create X & y

In [165]:
X = df["clean_text"] 
y = df["nation"]

print("Number of samples:", len(X))
print("Number of unique nations:", y.nunique())

Number of samples: 1574
Number of unique nations: 8


### Test/train split

In [166]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42,stratify=y)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

print("Train label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Training examples: 1259
Testing examples: 315
Train label distribution:
nation
north_american_native    296
greek                    243
german                   222
french                   134
indian                   104
english                   96
italian                   83
chinese                   81
Name: count, dtype: int64

Test label distribution:
nation
north_american_native    74
greek                    61
german                   55
french                   34
indian                   26
english                  24
italian                  21
chinese                  20
Name: count, dtype: int64


### Convert text to numbers and feel to model

In [167]:
baseline_model = Pipeline([
    ("vectorizer", CountVectorizer( max_features=20000, stop_words="english")),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

### Train Model

In [168]:
baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](8,)","['chinese','english','french',...,'indian','italian', 'north_american_native']"
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",20000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [169]:
print(baseline_model.named_steps['classifier'].coef_)
print(baseline_model.named_steps['classifier'].intercept_)

[[-6.14585243e-05 -1.77648474e-04  3.58274321e-04 ... -5.17576729e-04
   2.86267824e-06 -7.47730422e-04]
 [ 3.14958328e-06 -6.47239146e-04 -1.22803582e-03 ... -2.58929081e-04
   2.55125858e-06 -1.83574733e-04]
 [-6.46821960e-04  1.31905008e-04  1.87193382e-03 ...  3.59144275e-06
   2.30109582e-06 -2.84594961e-03]
 ...
 [ 2.16209642e-06  2.00949495e-03 -2.11493339e-03 ...  3.26153729e-06
   2.69690274e-06 -7.13266899e-04]
 [ 4.37427192e-06 -1.17301042e-03  2.92563538e-03 ...  3.16300733e-06
   2.12497082e-06 -7.77960409e-04]
 [ 6.52799799e-06 -3.74513101e-05 -8.53626810e-04 ... -2.91727923e-05
  -1.86870920e-05 -2.13697704e-03]]
[-1.89320419 -1.83023486  1.38849079 -0.45782184  5.15813923 -2.18035598
 -0.45440674  0.26939361]


## Predict and Evaluate

In [170]:
# Predictions on training data
y_train_pred = baseline_model.predict(X_train)

# Predictions on test data
y_test_pred = baseline_model.predict(X_test)

In [171]:
# Scores
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_macro_f1 = f1_score(y_train, y_train_pred, average="macro")
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

print("Train Macro F1:", train_macro_f1)
print("Test Macro F1:", test_macro_f1)

Train Accuracy: 1.0
Test Accuracy: 0.7523809523809524
Train Macro F1: 1.0
Test Macro F1: 0.7256254774858382


In [172]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df


Classification Report:
                       precision    recall  f1-score   support

              chinese       0.94      0.75      0.83        20
              english       0.83      0.62      0.71        24
               french       0.67      0.53      0.59        34
               german       0.73      0.69      0.71        55
                greek       0.66      0.87      0.75        61
               indian       0.90      0.69      0.78        26
              italian       0.53      0.48      0.50        21
north_american_native       0.82      0.92      0.87        74

             accuracy                           0.75       315
            macro avg       0.76      0.69      0.72       315
         weighted avg       0.75      0.75      0.74       315



,chinese,english,french,german,greek,indian,italian,north_american_native
chinese,15,0,0,1,2,1,0,1
english,0,15,0,4,2,0,0,3
french,0,2,18,4,3,0,4,3
german,0,0,2,38,10,0,3,2
greek,0,1,2,1,53,0,1,3
indian,0,0,1,2,3,18,0,2
italian,0,0,3,2,4,1,10,1
north_american_native,1,0,1,0,3,0,1,68


## Experiment 2: TF-IDF Vectorizer + Logistic Regression

In [173]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

tfidf_model = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=15000,
        stop_words="english"
    )),
    ("classifier", LogisticRegression(
        max_iter=1500,
        random_state=42
    ))
])

tfidf_model.fit(X_train, y_train)

y_train_pred = tfidf_model.predict(X_train)
y_test_pred = tfidf_model.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("Train Macro F1:", f1_score(y_train, y_train_pred, average="macro"))
print("Test Macro F1:", f1_score(y_test, y_test_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Train Accuracy: 0.9301032565528197
Test Accuracy: 0.6984126984126984
Train Macro F1: 0.9213698278307345
Test Macro F1: 0.6225447593615403

Classification Report:
                       precision    recall  f1-score   support

              chinese       1.00      0.50      0.67        20
              english       1.00      0.25      0.40        24
               french       0.85      0.32      0.47        34
               german       0.67      0.85      0.75        55
                greek       0.63      0.84      0.72        61
               indian       0.83      0.58      0.68        26
              italian       1.00      0.33      0.50        21
north_american_native       0.66      0.99      0.79        74

             accuracy                           0.70       315
            macro avg       0.83      0.58      0.62       315
         weighted avg       0.76      0.70      0.67       315



In [174]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

C_values = [2.0, 1.0, 0.7, 0.5, 0.3, 0.1]

results = []
models = {}

for C in C_values:
    model = Pipeline([
        ("vectorizer", CountVectorizer(
            max_features=10000,
            stop_words="english"
        )),
        ("classifier", LogisticRegression(
            C=C,
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_macro_f1 = f1_score(y_train, y_train_pred, average="macro")
    test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

    results.append({
        "C": C,
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "test_accuracy": accuracy_score(y_test, y_test_pred),
        "train_macro_f1": train_macro_f1,
        "test_macro_f1": test_macro_f1,
        "f1_gap": train_macro_f1 - test_macro_f1
    })

    models[C] = model

results_df = pd.DataFrame(results)
results_df.sort_values("test_macro_f1", ascending=False)

,C,train_accuracy,test_accuracy,train_macro_f1,test_macro_f1,f1_gap
5,0.1,0.998411,0.749206,0.998262,0.714905,0.283357
0,2.0,1.000000,0.742857,1.000000,0.714860,0.285140
1,1.0,1.000000,0.742857,1.000000,0.714860,0.285140
4,0.3,0.999206,0.746032,0.998986,0.713193,0.285793
2,0.7,1.000000,0.742857,1.000000,0.712884,0.287116
3,0.5,1.000000,0.742857,1.000000,0.712884,0.287116


In [175]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

min_df_values = [1, 2, 3, 5, 10]

results = []
models = {}

for min_df in min_df_values:
    model = Pipeline([
        ("vectorizer", CountVectorizer(
            max_features=15000,
            stop_words="english",
            min_df=min_df
        )),
        ("classifier", LogisticRegression(
            max_iter=1500,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_macro_f1 = f1_score(y_train, y_train_pred, average="macro")
    test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

    results.append({
        "min_df": min_df,
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "test_accuracy": accuracy_score(y_test, y_test_pred),
        "train_macro_f1": train_macro_f1,
        "test_macro_f1": test_macro_f1,
        "f1_gap": train_macro_f1 - test_macro_f1
    })

    models[min_df] = model

min_df_results = pd.DataFrame(results)
min_df_results.sort_values("test_macro_f1", ascending=False)

,min_df,train_accuracy,test_accuracy,train_macro_f1,test_macro_f1,f1_gap
0,1,1.0,0.749206,1.0,0.723268,0.276732
1,2,1.0,0.749206,1.0,0.716127,0.283873
2,3,1.0,0.746032,1.0,0.713852,0.286148
3,5,1.0,0.749206,1.0,0.713480,0.286520
4,10,1.0,0.742857,1.0,0.701652,0.298348


In [176]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

bigram_model = Pipeline([
    ("vectorizer", CountVectorizer(
        max_features=15000,
        stop_words="english",
        min_df=1,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1500,
        random_state=42
    ))
])

bigram_model.fit(X_train, y_train)

y_train_pred = bigram_model.predict(X_train)
y_test_pred = bigram_model.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("Train Macro F1:", f1_score(y_train, y_train_pred, average="macro"))
print("Test Macro F1:", f1_score(y_test, y_test_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Train Accuracy: 1.0
Test Accuracy: 0.746031746031746
Train Macro F1: 1.0
Test Macro F1: 0.7203941114298338

Classification Report:
                       precision    recall  f1-score   support

              chinese       0.94      0.75      0.83        20
              english       0.88      0.62      0.73        24
               french       0.69      0.53      0.60        34
               german       0.75      0.69      0.72        55
                greek       0.64      0.87      0.74        61
               indian       0.86      0.69      0.77        26
              italian       0.56      0.48      0.51        21
north_american_native       0.82      0.92      0.87        74

             accuracy                           0.75       315
            macro avg       0.77      0.69      0.72       315
         weighted avg       0.76      0.75      0.74       315



I tested several text representation and regularization settings:

- CountVectorizer with different vocabulary sizes

- TF-IDF features

- Logistic regression regularization through `C`

- Rare-word filtering using `min_df`

- Unigrams plus bigrams

The best overall baseline was CountVectorizer with 15,000 features and LogisticRegression, achieving approximately 0.75 accuracy and 0.72 macro F1 on the test set.

Although the model still overfits the training data, the test performance indicates that the dataset contains useful lexical signal. Since the goal of MythicLens is not only prediction but interpretable text exploration, the next step is to inspect learned feature weights and surface the top predictive words for each folktale label.